# GoldWork_Incremental
- Incremental Gold processing plus latest and timestamped volume snapshots

# Step 1  -- Imports and Setup
This cell imports The required helpers, switches to the right catlog, makes sure the gold schema exists and creates

- a `gold_run_id`
- a run date string
- a run timestamp string

Thease are used for tracking and snapshot publishing

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:
spark.sql("use catalog venk_novacart")
spark.sql("create schema if not exists gold_schema")
gold_run_id = str(uuid.uuid4())


run_ts_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

run_date_str = datetime.now().strftime("%Y-%m-%d")

print("current Gold Run ID",run_ts_str)
print("Run Timestamp Folder",run_date_str)

# Step -2 -- Gold control table
This table stores the latest Gold processing State

it tells Gold
- which silver data was processed last time
- how many gold rows were merged in last run

In [0]:
spark.sql("""
create table if not exists gold_schema.processing_control (
    layer string,
    entity_name string,
    last_processed_silver_run_id string,
    last_processed_silver_run_ts timestamp,
    rows_merged bigint,
    run_status string,
    gold_run_id string,
    updated_at timestamp
)
using delta
""")

# Step 3 --helper functions
This cell defines reusable Gold functions:
- upsert_to_gold() merges data into Gold current state tables
- get_last_processed_silver_ts() reads gold watermark from control table
- upsert_gold_control() updates Gold control after a sucessful run

In [0]:
def upsert_to_gold(df_source, target_table, join_key):
    if spark.catalog.tableExists(target_table):
        dt = DeltaTable.forName(spark, target_table)
        (dt.alias("target")
           .merge(df_source.alias("source"),
            f"target.{join_key} = source.{join_key}")
           .whenMatchedUpdateAll()
           .whenNotMatchedInsertAll()
           .execute())
    else:
        df_source.write.format("delta").saveAsTable(target_table)

In [0]:
def get_last_processed_silver_ts(entity_name: str):
    ctrl = (
        spark.table("venk_novacart.silver_schema.processing_control")
        .filter(
            (F.col("layer") == "gold") &
            (F.col("entity_name") == entity_name) &
            (F.col("run_status") == "SUCCESS")
        )
        .orderBy(F.col("updated_at").desc())
        .limit(1)
    )

    rows = ctrl.collect()
    if not rows:
        return None

    return rows[0]["last_processed_silver_run_ts"]

In [0]:
def upsert_gold_control(entity_name, last_processed_silver_run_id, last_processed_run_ts, rows_merged):
    ctrl_df = spark.createDataFrame(
        [(
            "gold",
            entity_name,
            last_processed_silver_run_id,
            last_processed_run_ts,
            int(rows_merged),
            "SUCCESS",
            gold_run_id,
            datetime.utcnow()
        )],
        schema="""
        layer string,
        entity_name string,
        last_processed_silver_run_id string,
        last_processed_silver_run_ts timestamp,
        rows_merged bigint,
        run_status string,
        gold_run_id string,
        updated_at timestamp
        """
    )

    dt = DeltaTable.forName(spark, "gold_schema.processing_control") .alias('t') .merge(ctrl_df.alias("s"), "t.layer = s.layer and t.entity_name = s.entity_name") .whenMatchedUpdate(set={
            "last_processed_silver_run_id": "s.last_processed_silver_run_id",
            "last_processed_silver_run_ts": "s.last_processed_silver_run_ts",
            "rows_merged": "s.rows_merged",
            "run_status": "s.run_status",
            "gold_run_id": "s.gold_run_id",
            "updated_at": "s.updated_at"
        }) .whenNotMatchedInsertAll() .execute()
        

# Step 4 -- Read changed Silver rows only

This cell Reads the full current state tables but filters out only the rows that changed since the last Gold run.
This is the starting point for Gold incremntal processing.

In [0]:
last_gold_ts = get_last_processed_silver_ts("orders_information")

print("Last Processed Silver Timestamp for Gold = ", last_gold_ts)

silver_orders_current = spark.read.table("venk_novacart.silver_schema.orders_transformed")
silver_products_current = spark.read.table("venk_novacart.silver_schema.products_transformed")
silver_payments_current = spark.read.table("venk_novacart.silver_schema.payments_transformed")

if last_gold_ts is None:

    changed_orders = silver_orders_current
    changed_products = silver_products_current
    changed_payments = silver_payments_current

else:

    changed_orders = silver_orders_current.filter(F.col("updated_at") > F.lit(last_gold_ts))
    changed_products = silver_products_current.filter(F.col("updated_at") > F.lit(last_gold_ts))
    changed_payments = silver_payments_current.filter(F.col("updated_at") > F.lit(last_gold_ts))

changed_orders_count = changed_orders.count()
changed_products_count = changed_products.count()
changed_payments_count = changed_payments.count()

print(f"Number of changed orders = {changed_orders_count}")
print(f"Number of changed products = {changed_products_count}")
print(f"Number of changed payments = {changed_payments_count}")

# Step 5 -- find impacted order IDs
Gold is built at **order grain**, so if anything changes in orders,products,or payments,we identify which order_id values are impacted.

Only those order IDs are rebuilt in Gold.

In [0]:
impacted_from_orders = changed_orders.select("order_id").distinct()
impacted_from_payments = changed_payments.select("order_id").distinct()
impacted_from_products =(
  changed_products.alias("p")
  .join(silver_orders_current.alias("o"), F.col("p.product_id") == F.col("o.product_id"), "inner")
  .select(F.col("o.order_id")).distinct()
)

impact_order_ids =(
  impacted_from_orders
  .union(impacted_from_payments)
  .union(impacted_from_products).distinct()
)
print("impacted_order_ids=",impact_order_ids.count())
display(impact_order_ids.orderBy("order_id"))

# Step 6 -- Build Gold delta for impacted orders

**This cell joins the impacted orders with the current Silver products and payments tables,derives business columns, and builds the Gold delta that will be merged into the Gold current -state table.**

In [0]:
impacted_orders = (
    silver_orders_current.alias("o")
    .join(impact_order_ids.alias("i"), ["order_id"], "inner")
)
gold_delta = (
    impacted_orders.alias("o")
    .join(
        silver_products_current.alias("p"),
        F.col("o.product_id") == F.col("p.product_id"),
        "inner",
    )
    .join(
        silver_payments_current.alias("pay"),
        F.col("o.order_id") == F.col("pay.order_id"),
        "inner",
    )
    .select(
        F.col("o.order_id"),
        F.col("o.customer_id"),
        F.col("p.product_id"),
        F.col("p.product_name"),
        F.col("p.category"),
        F.col("p.price").alias("product_price"),
        F.col("o.order_amount"),
        F.col("o.order_status"),
        F.col("pay.payment_id"),
        F.col("pay.payment_status"),
        F.col("pay.paid_amount"),
        F.col("o.order_date"),
        F.col("o.order_year"),
        F.col("o.order_month"),
        F.greatest(
            F.col("o.updated_at").cast("timestamp"),
            F.col("p.updated_at").cast("timestamp"),
            F.col("pay.processed_at").cast("timestamp")
        ).alias("gold_update_ts")
    )
    .dropDuplicates(["order_id"])
    .withColumn(
        "payment_completion_ratio",
        F.when(
            F.col("order_amount") > 0,
            F.col("paid_amount") / F.col("order_amount")
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "payment_status",
        F.when(F.col("order_amount") == 0, "Invalid_order_amount")
        .when(F.col("payment_completion_ratio") == 1, "Paid")
        .when(F.col("payment_completion_ratio") == 0, "Unpaid")
        .when(F.col("payment_completion_ratio") > 1, "Overpaid")
        .when(F.col("payment_completion_ratio") < 1, "Partially_paid")
    )
    .withColumn("gold_updated_date", F.to_date(F.col("gold_update_ts")))
    .withColumn("gold_run_id", F.lit(gold_run_id))
)
print("gold_delta_rows=", gold_delta.count())
display(gold_delta)

# Step 7 -- Merge Gold current-state table

if Gold delta contains rows, this cell merges them into `gold_schema.orders_information`.
if there are no impacted rows,nothing is merged.

In [0]:
# Only upsert if there are new or changed records to avoid unnecessary writes and empty files in Delta table
if gold_delta.count() > 0:
    upsert_to_gold(gold_delta, "venk_novacart.gold_schema.orders_information", "order_id")
else:
    print("No new records to be inserted in gold table")

In [0]:
%sql
select * from venk_novacart.gold_schema.orders_information

# Step 8 -- Maintain Gold SCD Type 2 history

If a current Gold row changes,the old version is closed(is_current = false) and a new current version is inserted.

In [0]:
# create table (unchanged)
if not spark.catalog.tableExists("venk_novacart.gold_schema.orders_information_scd2"):
    spark.sql("""
        CREATE TABLE venk_novacart.gold_schema.orders_information_scd2
        USING DELTA
        AS
        SELECT *,
               CAST(NULL AS TIMESTAMP) AS valid_from_ts,
               CAST(NULL AS TIMESTAMP) AS valid_to_ts,
               TRUE AS is_current
        FROM venk_novacart.gold_schema.orders_information
        WHERE 1 = 0
    """)

# run only if data exists
if gold_delta.count() > 0:

    gold_delta.createOrReplaceTempView("gold_delta_view")

    # expire old records
    spark.sql("""
        MERGE INTO venk_novacart.gold_schema.orders_information_scd2 t
        USING gold_delta_view s
        ON t.order_id = s.order_id AND t.is_current = true

        WHEN MATCHED AND (
            NOT (t.order_status <=> s.order_status) OR
            NOT (t.order_amount <=> s.order_amount) OR
            NOT (t.paid_amount <=> s.paid_amount) OR
            NOT (t.payment_id <=> s.payment_id) OR
            NOT (t.category <=> s.category) OR
            NOT (t.product_name <=> s.product_name) OR
            NOT (t.product_price <=> s.product_price)
        )
        THEN UPDATE SET
            t.is_current = false,
            t.valid_to_ts = s.gold_update_ts
    """)

    # insert new + changed records
    spark.sql("""
        INSERT INTO venk_novacart.gold_schema.orders_information_scd2
        SELECT
            s.*,
            s.gold_update_ts AS valid_from_ts,
            CAST(NULL AS TIMESTAMP) AS valid_to_ts,
            TRUE AS is_current
        FROM gold_delta_view s
        LEFT JOIN venk_novacart.gold_schema.orders_information_scd2 t
            ON t.order_id = s.order_id AND t.is_current = true
        WHERE t.order_id IS NULL
           OR (
                NOT (t.order_status <=> s.order_status) OR
                NOT (t.order_amount <=> s.order_amount) OR
                NOT (t.paid_amount <=> s.paid_amount) OR
                NOT (t.payment_id <=> s.payment_id) OR
                NOT (t.category <=> s.category) OR
                NOT (t.product_name <=> s.product_name) OR
                NOT (t.product_price <=> s.product_price)
           )
    """)
                     
    
    

# Step 9 -- Update category-level gold aggregation
this cell recalculates category-level business matrics only for categories impacted in the current run,then merges them into the category performance Gold table

In [0]:


if gold_delta.count() > 0:

    impacted_categories = (
        gold_delta
        .select("category")
        .filter(F.col("category").isNotNull())
        .distinct()
    )

    category_perf_delta = (
        spark.read.table("venk_novacart.gold_schema.orders_information")

        # Keep only impacted categories
        .join(impacted_categories, "category", "inner").groupBy("category")
        .agg(

            # Total unique orders
            F.countDistinct("order_id").alias("total_orders"),

            # Gross Merchandise Value (only positive amounts)
            F.sum(
                F.when(F.col("order_amount") > 0, F.col("order_amount"))
                 .otherwise(F.lit(0.0))
            ).alias("Gross_Merchandise_Value"),

            # Total Paid Amount (only positive values)
            F.sum(
                F.when(F.col("paid_amount") > 0, F.col("paid_amount"))
                 .otherwise(F.lit(0.0))
            ).alias("Total_Paid_Amount"),

            # Average payment completion ratio
            F.avg(F.col("payment_completion_ratio"))
             .alias("Average_Payment_Completion_Ratio"),

            # Payment failure rate = failed / total
            (
                F.sum(
                    F.when(F.col("payment_status") == "FAILED", 1)
                     .otherwise(0)
                ) / F.count("*")
            ).alias("Payment_Failure_Rate")
        )
    )
upsert_to_gold(category_perf_delta, "venk_novacart.gold_schema.category_performance","category")

In [0]:
%sql
select * from venk_novacart.gold_schema.category_performance

# Step 10 -- publish Gold snapshot to volume
This cell writes two kinds of Gold outputs to a Databricks volume:

**latest Snapshot ** over written every sucessful run

**timestamped historical snapshot** a new folder for each sucessful run

This is usefull for audit,rollback,and teaching demos.

In [0]:
spark.sql("create volume if not exists venk_novacart.gold_schema.gold_snapshots_vol")

In [0]:
# -----------------------------------------
# Define paths
# -----------------------------------------

latest_orders_path = "/Volumes/venk_novacart/gold_schema/gold_snapshots_vol/gold_latest/orders_information"

latest_category_path = "/Volumes/venk_novacart/gold_schema/gold_snapshots_vol/gold_latest/category_performance"

historical_orders_path = f"/Volumes/venk_novacart/gold_schema/gold_snapshots_vol/gold_snapshots/orders_information/run_date={run_date_str}/run_ts={run_ts_str}"

historical_category_path = f"/Volumes/venk_novacart/gold_schema/gold_snapshots_vol/gold_snapshots/category_performance/run_date={run_date_str}/run_ts={run_ts_str}"


# -----------------------------------------
# Read tables once
# -----------------------------------------

orders_df = spark.read.table("venk_novacart.gold_schema.orders_information")

category_df = spark.read.table("venk_novacart.gold_schema.category_performance")


# -----------------------------------------
# Write latest (overwrite)
# -----------------------------------------

orders_df.write.mode("overwrite").format("parquet").save(latest_orders_path)

category_df.write.mode("overwrite").format("parquet").save(latest_category_path)


# -----------------------------------------
# Write historical (snapshot)
# -----------------------------------------

orders_df.write.mode("overwrite").format("parquet").save(historical_orders_path)

category_df.write.mode("overwrite").format("parquet").save(historical_category_path)


# -----------------------------------------
# Print paths
# -----------------------------------------

print("Latest Orders Path:", latest_orders_path)
print("Latest Category Path:", latest_category_path)
print("Historical Orders Path:", historical_orders_path)
print("Historical Category Path:", historical_category_path)

Step -- 11 Update 

In [0]:
from pyspark.sql import functions as F

# Get latest timestamp
latest_silver_ts = (
    silver_orders_current
    .agg(F.max("bronze_ingested_at").alias("mx"))
    .first()["mx"]
)

# Get latest run_id for that timestamp
latest_silver_run_id = (
    silver_orders_current
    .filter(F.col("bronze_ingested_at") == latest_silver_ts)
    .agg(F.max("silver_run_id").alias("mx"))
    .first()["mx"]
) if latest_silver_ts is not None else None

# Upsert into control table
upsert_gold_control(
    "orders_information",
    latest_silver_run_id,
    latest_silver_ts,
    gold_delta.count()
)

# Display control table
display(
    spark.table("venk_novacart.gold_schema.processing_control")
)